# Semana 13: La Navaja Suiza - Descomposición en Valores Singulares (SVD)
## Notebook de Ejercicios (NB2) - Sistema de Recomendación con MovieLens

### Propósito de la sesión
Aplicar la **descomposición en valores singulares (SVD)** para construir un sistema de recomendación de películas. Trabajaremos con el dataset **MovieLens** (versión pequeña), construiremos la matriz usuario-película con valores faltantes, utilizaremos SVD para predecir ratings no vistos y generaremos recomendaciones personalizadas para un usuario.

### Objetivos de aprendizaje
*   Cargar y explorar el dataset **MovieLens** (ratings de películas).
*   Construir la **matriz usuario-película** con `NaN` para ratings no observados.
*   Aplicar **SVD truncada** para aproximar la matriz y predecir ratings faltantes.
*   Evaluar la calidad de las predicciones (opcional).
*   Generar **recomendaciones** para un usuario basadas en las predicciones.
*   Conectar con aplicaciones reales de filtrado colaborativo (Netflix Prize, etc.).

## Configuración Inicial

Importamos las librerías necesarias: pandas para manipulación de datos, numpy para operaciones, matplotlib para visualizaciones, y surprise (opcional) para comparación.

In [ ]:
# Importamos librerías
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.sparse.linalg import svds
from sklearn.metrics import mean_squared_error

# Configuración de matplotlib
plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams['figure.figsize'] = (12, 6)

# Fijamos semilla para reproducibilidad
np.random.seed(42)

print("🎯 Librerías importadas correctamente.")

---
## 1. Carga y Exploración del Dataset MovieLens

Utilizaremos la versión pequeña de MovieLens (100k ratings) disponible en línea. La descargaremos directamente desde la URL.

In [ ]:
# URLs de los datos
url_ratings = 'https://files.grouplens.org/datasets/movielens/ml-latest-small/ratings.csv'
url_movies = 'https://files.grouplens.org/datasets/movielens/ml-latest-small/movies.csv'

# Cargamos los datos
ratings = pd.read_csv(url_ratings)
movies = pd.read_csv(url_movies)

print("🔷 Datos de ratings:")
print(f"  Shape: {ratings.shape}")
print(f"  Columnas: {ratings.columns.tolist()}")
print(ratings.head())

print("\n🔶 Datos de películas:")
print(f"  Shape: {movies.shape}")
print(f"  Columnas: {movies.columns.tolist()}")
print(movies.head())

In [ ]:
# Estadísticas básicas
print("📊 Estadísticas de ratings:")
print(f"  Número de usuarios únicos: {ratings['userId'].nunique()}")
print(f"  Número de películas únicas: {ratings['movieId'].nunique()}")
print(f"  Rango de ratings: {ratings['rating'].min()} - {ratings['rating'].max()}")
print(f"  Media de ratings: {ratings['rating'].mean():.2f}")
print(f"  Desviación estándar: {ratings['rating'].std():.2f}")

# Distribución de ratings
plt.figure(figsize=(10, 5))
plt.subplot(1, 2, 1)
sns.histplot(ratings['rating'], bins=10, kde=True)
plt.title('Distribución de ratings')

plt.subplot(1, 2, 2)
ratings_per_user = ratings.groupby('userId').size()
sns.histplot(ratings_per_user, bins=30, kde=True)
plt.title('Número de ratings por usuario')
plt.xlabel('Ratings por usuario')
plt.tight_layout()
plt.show()

---
## Actividad 1: Construir la matriz usuario-película

Construimos una matriz donde las filas son usuarios, las columnas son películas, y las entradas son los ratings. Los valores faltantes se representan con `NaN`.

In [ ]:
# Creamos la matriz usuario-película (pivot table)
user_movie_matrix = ratings.pivot(index='userId', columns='movieId', values='rating')

print(f"Matriz usuario-película shape: {user_movie_matrix.shape}")
print(f"Número de valores no nulos: {user_movie_matrix.count().sum()}")
print(f"Densidad de la matriz: {user_movie_matrix.count().sum() / (user_movie_matrix.shape[0] * user_movie_matrix.shape[1]):.4%}")

# Mostramos una pequeña porción
user_movie_matrix.iloc[:5, :5]

In [ ]:
# Para trabajar con SVD, necesitamos una matriz densa sin NaNs.
# Una estrategia simple es llenar los NaNs con 0 (o con la media global).
# En la práctica, se usan técnicas más sofisticadas, pero aquí lo haremos así.

# Opción 1: llenar con 0
R = user_movie_matrix.fillna(0).values

# Opción 2: llenar con la media global (mejor)
global_mean = ratings['rating'].mean()
R_mean_filled = user_movie_matrix.fillna(global_mean).values

print(f"Matriz R (con ceros) shape: {R.shape}")
print(f"Matriz R_mean_filled shape: {R_mean_filled.shape}")
print(f"Media global de ratings: {global_mean:.4f}")

---
## Actividad 2: Aplicar SVD para completar la matriz

Utilizamos SVD truncada (solo los primeros $k$ valores singulares) para aproximar la matriz y predecir ratings faltantes.

$$\mathbf{R} \approx \mathbf{U}_k \boldsymbol{\Sigma}_k \mathbf{V}_k^\top$$

Luego, la matriz completa (con predicciones) es $\hat{\mathbf{R}} = \mathbf{U}_k \boldsymbol{\Sigma}_k \mathbf{V}_k^\top$.

In [ ]:
def svd_recommendation(R, k=20):
    """
    Aplica SVD truncada a la matriz R y devuelve la matriz reconstruida.
    """
    # Centramos los datos (restando la media de cada usuario)
    user_means = np.mean(R, axis=1, keepdims=True)
    R_demeaned = R - user_means
    
    # SVD truncada
    U, sigma, Vt = svds(R_demeaned, k=k)
    
    # Convertimos sigma a matriz diagonal
    sigma = np.diag(sigma)
    
    # Reconstruimos
    R_hat = U @ sigma @ Vt + user_means
    
    return R_hat, U, sigma, Vt, user_means

# Probamos con diferentes valores de k
k = 20
R_hat, U, sigma, Vt, user_means = svd_recommendation(R_mean_filled, k=k)

print(f"Matriz reconstruida shape: {R_hat.shape}")
print(f"Rango de valores en R_hat: [{R_hat.min():.2f}, {R_hat.max():.2f}]")

In [ ]:
# Evaluación simple: error en los ratings observados
# Obtenemos las posiciones de los ratings no nulos en la matriz original
mask = user_movie_matrix.notna().values
observed_pred = R_hat[mask]
observed_true = user_movie_matrix.values[mask]

rmse = np.sqrt(mean_squared_error(observed_true, observed_pred))
print(f"RMSE en ratings observados (k={k}): {rmse:.4f}")

# Probemos con diferentes k
k_values = [5, 10, 20, 30, 50]
rmse_values = []

for k_test in k_values:
    R_hat_test, _, _, _, _ = svd_recommendation(R_mean_filled, k=k_test)
    pred_test = R_hat_test[mask]
    rmse_test = np.sqrt(mean_squared_error(observed_true, pred_test))
    rmse_values.append(rmse_test)
    print(f"k={k_test:2d}, RMSE={rmse_test:.4f}")

plt.figure(figsize=(10, 5))
plt.plot(k_values, rmse_values, 'bo-', linewidth=2)
plt.xlabel('Número de componentes (k)')
plt.ylabel('RMSE')
plt.title('RMSE en ratings observados vs número de componentes SVD')
plt.grid(True)
plt.show()

---
## Actividad 3: Hacer recomendaciones para un usuario

Seleccionamos un usuario (por ejemplo, el usuario 1) y recomendamos las películas con mayor rating predicho que el usuario no ha visto.

In [ ]:
# Elegimos un usuario (por ejemplo, userId=1)
user_id = 1
user_idx = user_movie_matrix.index.get_loc(user_id)

# Películas que el usuario ya ha visto (ratings no nulos)
seen_movies = user_movie_matrix.loc[user_id].dropna().index
print(f"Usuario {user_id} ha visto {len(seen_movies)} películas.")

# Predicciones para este usuario
user_predictions = R_hat[user_idx, :]

# Enmascaramos las películas ya vistas (ponemos -inf para no recomendarlas)
mask_seen = user_movie_matrix.loc[user_id].notna().values
user_predictions_masked = user_predictions.copy()
user_predictions_masked[mask_seen] = -np.inf

# Obtenemos los índices de las películas con mayor predicción
top_n = 10
top_movie_indices = np.argsort(user_predictions_masked)[::-1][:top_n]
top_movie_ids = user_movie_matrix.columns[top_movie_indices]

# Información de las películas recomendadas
recommended_movies = movies[movies['movieId'].isin(top_movie_ids)].copy()
recommended_movies['predicted_rating'] = user_predictions[top_movie_indices]
recommended_movies = recommended_movies.sort_values('predicted_rating', ascending=False)

print(f"\n🎬 Top {top_n} recomendaciones para el usuario {user_id}:")
display(recommended_movies[['title', 'genres', 'predicted_rating']])

In [ ]:
# Verificamos qué películas ha visto el usuario y sus ratings reales
user_ratings = ratings[ratings['userId'] == user_id].merge(movies, on='movieId')
user_ratings = user_ratings.sort_values('rating', ascending=False)

print(f"🔍 Películas ya vistas por el usuario {user_id} (top 5 por rating):")
display(user_ratings[['title', 'genres', 'rating']].head())

### 3.1. Interpretación de los factores latentes

Las matrices $\mathbf{U}$ y $\mathbf{V}$ obtenidas de la SVD tienen interpretación:
*   Las filas de $\mathbf{U}$ (o columnas en nuestra implementación) representan a los usuarios en el espacio de "factores latentes" (géneros, estilos, etc.).
*   Las filas de $\mathbf{V}^\top$ representan a las películas en ese mismo espacio.
*   El producto punto entre el vector de un usuario y el de una película da la predicción del rating.

Podemos explorar estos factores para entender qué representan.

In [ ]:
# Extraemos los factores de películas (Vt.T)
movie_factors = Vt.T  # shape: (n_movies, k)

# Para una película específica, por ejemplo 'Toy Story' (movieId=1)
toy_story_idx = movies[movies['title'].str.contains('Toy Story', na=False)].index[0]
toy_story_id = movies.loc[toy_story_idx, 'movieId']
toy_story_col = user_movie_matrix.columns.get_loc(toy_story_id)
toy_story_factors = movie_factors[toy_story_col]

print(f"Factores latentes para 'Toy Story':")
print(toy_story_factors)

# Buscamos películas con factores similares (vecinos más cercanos en espacio latente)
from sklearn.metrics.pairwise import cosine_similarity

similarities = cosine_similarity(movie_factors, movie_factors[toy_story_col].reshape(1, -1)).flatten()
similar_indices = np.argsort(similarities)[::-1][1:11]  # top 10 (excluyendo la misma)

similar_movies = movies.iloc[similar_indices].copy()
similar_movies['similarity'] = similarities[similar_indices]

print("\n🎥 Películas similares a 'Toy Story' en el espacio latente:")
display(similar_movies[['title', 'genres', 'similarity']])

---
## Ejercicios para el Estudiante

1.  **Diferentes estrategias de imputación:** Prueba llenar los valores faltantes con la media de cada usuario en lugar de la media global. ¿Cómo afecta al RMSE y a las recomendaciones?

2.  **Evaluación en test:** Divide los datos en entrenamiento y prueba (ocultando algunos ratings aleatoriamente). Evalúa el RMSE en el conjunto de prueba. ¿Qué k da el mejor resultado?

3.  **Recomendaciones para otro usuario:** Elige un usuario con pocas valoraciones y observa las recomendaciones. ¿Tienen sentido?

4.  **Análisis de factores:** Intenta interpretar los factores latentes mirando qué películas tienen valores extremos en cada factor. ¿Corresponden a géneros? (Por ejemplo, factor 1: películas de acción vs. drama).

5.  **Comparación con surprise:** Usa la librería `surprise` (scikit-surprise) que implementa SVD y otros algoritmos de recomendación. Compara tus resultados con los de la librería.

6.  **Reflexión:** ¿Qué ventajas y desventajas tiene usar SVD para sistemas de recomendación? ¿Cómo maneja el problema de cold-start (nuevos usuarios o películas)?

---
## Conclusiones de la Sesión

*   Construimos la **matriz usuario-película** a partir del dataset MovieLens, observando su alta escasez (sparsity).
*   Aplicamos **SVD truncada** para aproximar la matriz y predecir ratings faltantes, utilizando la técnica de restar la media por usuario.
*   Evaluamos el error (RMSE) en los ratings observados para diferentes valores de $k$, observando que un $k$ moderado (20-30) da buenos resultados.
*   Generamos **recomendaciones personalizadas** para un usuario seleccionando las películas con mayor rating predicho no vistas.
*   Exploramos los **factores latentes** obtenidos, encontrando que capturan similitudes entre películas (por ejemplo, encontrar películas similares a "Toy Story").
*   Conectamos este ejercicio con aplicaciones reales: el **Netflix Prize** popularizó el uso de SVD y factorización de matrices para sistemas de recomendación.

¡Ahora sabes cómo construir un sistema de recomendación basado en SVD!